# 01 贝叶斯定理

问题与建模思路参考 Allen B. Downey *Think Bayes*（中译《贝叶斯思维》）第 1 章。

**学习目标**：
- 区分条件概率、联合概率与贝叶斯更新
- 用「先验 × 似然 → 归一化」手算曲奇饼、M&M、Monty Hall
- 把同一套路写成可运行的 `dict` 计算


## 1. 条件概率与联合概率

事件 $A$、$B$：

$$
P(A \mid B) = \frac{P(A \cap B)}{P(B)}
$$

联合概率：

$$
P(A \cap B) = P(B)\,P(A \mid B) = P(A)\,P(B \mid A)
$$

由此得到**贝叶斯定理**：

$$
P(H \mid D) = \frac{P(H)\,P(D \mid H)}{P(D)}
$$

其中 $P(H)$ 为先验，$P(D \mid H)$ 为似然，$P(D)=\sum_h P(h)P(D\mid h)$ 为证据（归一化常数）。


In [2]:
def bayes_update(prior: dict, likelihood: dict) -> dict:
    """posterior ∝ prior × likelihood, then normalize."""
    unnorm = {h: prior[h] * likelihood[h] for h in prior}
    z = sum(unnorm.values())
    return {h: v / z for h, v in unnorm.items()}


def show(dist: dict, title: str = ""):
    if title:
        print(title)
    for h, p in dist.items():
        print(f"  {h}: {p:.4f}")


## 2. 曲奇饼问题

两碗曲奇饼：

| 碗 | 香草 | 巧克力 |
|----|------|--------|
| Bowl 1 | 30 | 10 |
| Bowl 2 | 20 | 20 |

随机选一碗，再随机抽一块，得到**香草**。问来自 Bowl 1 的后验？

先验 $P(B_1)=P(B_2)=1/2$，似然 $P(V\mid B_1)=3/4$，$P(V\mid B_2)=1/2$。


In [4]:
prior = {"Bowl1": 0.5, "Bowl2": 0.5}
likelihood_vanilla = {"Bowl1": 30 / 40, "Bowl2": 20 / 40}
posterior = bayes_update(prior, likelihood_vanilla)
show(posterior, "抽到香草后的后验")
assert abs(posterior["Bowl1"] - 0.6) < 1e-9
print("Bowl1 后验 = 0.6 ✓")


抽到香草后的后验
  Bowl1: 0.6000
  Bowl2: 0.4000
Bowl1 后验 = 0.6 ✓


## 3. M&M 豆问题

1994 与 1996 年袋中颜色比例不同。从两袋各抽一颗：一颗是黄色、一颗是绿色。
假设 $A$：黄来自 1994 袋、绿来自 1996 袋；$B$ 相反。先验均等。

| 颜色 | 1994 | 1996 |
|------|------|------|
| 黄 | 20% | 14% |
| 绿 | 10% | 20% |

似然：$P(D\mid A)=0.20\times0.20$，$P(D\mid B)=0.14\times0.10$。


In [5]:
prior_mm = {"A_yellow94": 0.5, "B_yellow96": 0.5}
likelihood_mm = {
    "A_yellow94": 0.20 * 0.20,  # yellow from 94, green from 96
    "B_yellow96": 0.14 * 0.10,  # yellow from 96, green from 94
}
posterior_mm = bayes_update(prior_mm, likelihood_mm)
show(posterior_mm, "M&M 后验")
print(f"P(A|D) ≈ {posterior_mm['A_yellow94']:.4f}")


M&M 后验
  A_yellow94: 0.7407
  B_yellow96: 0.2593
P(A|D) ≈ 0.7407


## 4. Monty Hall 难题

三扇门，奖品均匀随机。你选门 1；Monty 打开门 3（山羊）。应否换到门 2？

假设 $H_i$：奖品在门 $i$。先验 $P(H_i)=1/3$。

数据 $D$：Monty 开了门 3。约定：Monty 总开山羊门；若有两扇可选则随机开。

$$
P(D \mid H_1)=\tfrac12,\quad P(D \mid H_2)=1,\quad P(D \mid H_3)=0
$$


In [6]:
prior_mh = {"door1": 1 / 3, "door2": 1 / 3, "door3": 1 / 3}
# 你选门 1，Monty 开门 3
likelihood_mh = {"door1": 0.5, "door2": 1.0, "door3": 0.0}
posterior_mh = bayes_update(prior_mh, likelihood_mh)
show(posterior_mh, "Monty 开门 3 后的后验")
assert abs(posterior_mh["door1"] - 1 / 3) < 1e-9
assert abs(posterior_mh["door2"] - 2 / 3) < 1e-9
print("换门胜率 2/3 ✓")


Monty 开门 3 后的后验
  door1: 0.3333
  door2: 0.6667
  door3: 0.0000
换门胜率 2/3 ✓


## 5. 历时诠释（Diachronic）

贝叶斯更新可写成：

$$
\underbrace{P(H \mid D)}_{\text{后验}}
= \frac{\overbrace{P(H)}^{\text{先验}}\;\overbrace{P(D \mid H)}^{\text{似然}}}
{\underbrace{P(D)}_{\text{证据}}}
$$

同一套公式适用于「一次性」谜题（Monty Hall）与「随数据不断修正信念」的估计问题（后续章节）。


In [7]:
# 曲奇饼：连抽两次香草（有放回近似 / 或按剩余计数）
# 这里用「有放回」演示多次 Update：似然连乘
prior2 = {"Bowl1": 0.5, "Bowl2": 0.5}
like_v = {"Bowl1": 0.75, "Bowl2": 0.5}
post1 = bayes_update(prior2, like_v)
post2 = bayes_update(post1, like_v)
show(post1, "第一次香草")
show(post2, "第二次香草（有放回）")


第一次香草
  Bowl1: 0.6000
  Bowl2: 0.4000
第二次香草（有放回）
  Bowl1: 0.6923
  Bowl2: 0.3077


## 小结

1. 先列假设与先验，再写每个假设下数据的似然，乘积归一化即得后验。
2. 曲奇饼 / M&M / Monty Hall 共用同一计算骨架；差别只在似然怎么定义。
3. 下一章把该骨架封装成 `Pmf` / `Suite`，避免每次手写 `dict`。
